In [1]:
import os
import re
import pandas as pd
from tqdm import tqdm
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)

RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

stemmer = PorterStemmer()
STANDARD_STOP = set(stopwords.words('english'))
CUSTOM_STOP_STEMMED = {
    'paper', 'propos', 'method', 'approach', 'result',
    'show', 'base', 'also', 'howev', 'therefor', 'thu',
    'furthermor', 'present', 'work', 'studi', 'experiment', 'evalu'
}

PATTERN_URL = re.compile(r'http\S+')
PATTERN_LATEX = re.compile(r'\$.*?\$')
PATTERN_CLEAN = re.compile(r'[^a-z0-9\s]')
PATTERN_SPACE = re.compile(r'\s+')

In [2]:
def preprocess_bm25(text: str) -> str:
    text = str(text).lower()
    text = PATTERN_URL.sub('', text)
    text = PATTERN_LATEX.sub(' math ', text)
    text = PATTERN_CLEAN.sub(' ', text)
    text = PATTERN_SPACE.sub(' ', text).strip()
    
    tokens = [t for t in text.split() if t not in STANDARD_STOP and len(t) > 2]
    stemmed_tokens = [stemmer.stem(t) for t in tokens]
    final_tokens = [t for t in stemmed_tokens if t not in CUSTOM_STOP_STEMMED]
    
    return ' '.join(final_tokens)

In [3]:
CORPUS_RAW_PATH = f"{RAW_DIR}/corpus.parquet"
CORPUS_PROCESSED_PATH = f"{PROCESSED_DIR}/corpus_bm25.parquet"

print("Dang doc du lieu tho...")
df = pd.read_parquet(CORPUS_RAW_PATH)

title = df['title'].fillna('')
abstract = df['text'].fillna('')
df['text'] = title + ' ' + title + ' ' + abstract

df = df.dropna(subset=['text']).reset_index(drop=True)

print("Dang chay tien xu ly...")
tqdm.pandas(desc="Processing BM25")
df['text_bm25'] = df['text'].progress_apply(preprocess_bm25)

df_final = df[['_id', 'title', 'text', 'text_bm25']].copy()
df_final = df_final.rename(columns={'_id': 'id'})

df_final.to_parquet(CORPUS_PROCESSED_PATH, index=False)
print(f"Da luu file thanh cong tai: {CORPUS_PROCESSED_PATH}")

Dang doc du lieu tho...
Dang chay tien xu ly...


Processing BM25: 100%|██████████| 25657/25657 [00:44<00:00, 576.74it/s]


Da luu file thanh cong tai: ../data/processed/corpus_bm25.parquet
